<a href="https://colab.research.google.com/github/taek20230541/maritimedatamining/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10주차 문제3

In [4]:
import pandas as pd
from scipy import stats

# 테스트용 데이터 생성 (파일이 없어도 실행됨)
data = {
    "branch": ["A", "A", "B", "B", "C", "C"],
    "score": [85, 90, 78, 82, 92, 88]
}
df = pd.DataFrame(data)

try:
    # 그룹핑 및 ANOVA 수행
    groups = [group["score"] for name, group in df.groupby("branch")]
    f_stat, p_val = stats.f_oneway(*groups)

    print("테스트 실행 성공!")
    print(f"F-statistic: {f_stat:.4f}, p-value: {p_val:.4f}")

except Exception as e:
    print(f"에러 발생 원인: {e}")

테스트 실행 성공!
F-statistic: 5.7018, p-value: 0.0951


# 10주차 문제4

In [6]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# 1. 테스트용 데이터 생성 (실제 파일이 있다면 pd.read_csv 사용)
# 파일이 없어서 생기는 에러를 방지하기 위해 가상 데이터를 만듭니다.
data = {
    'ad_type': ['video']*15 + ['image']*15,
    'age_group': (['20s']*5 + ['30s']*5 + ['40s']*5) * 2,
    'purchase_amount': [
        80, 85, 78, 92, 88, 70, 75, 72, 68, 71, 60, 58, 62, 65, 59,  # 영상광고 그룹
        65, 68, 62, 70, 66, 75, 78, 80, 82, 79, 85, 88, 90, 92, 89   # 이미지광고 그룹
    ]
}
df = pd.DataFrame(data)

# 2. 이원분산분석 모델 설정 및 수행
# 공식: 종속변수 ~ 독립변수1 * 독립변수2 (별표는 상호작용 효과를 포함한다는 뜻입니다)
model = ols('purchase_amount ~ C(ad_type) * C(age_group)', data=df).fit()
anova_result = anova_lm(model)

# 3. 결과 출력 및 가독성 정리
print(f"{' [이원분산분석 결과 보고서] ':=^50}")
print(anova_result)
print("-" * 50)

# 4. 주요 지표 해석 자동화
p_ad = anova_result.loc['C(ad_type)', 'PR(>F)']
p_age = anova_result.loc['C(age_group)', 'PR(>F)']
p_inter = anova_result.loc['C(ad_type) * C(age_group)', 'PR(>F)'] if 'C(ad_type):C(age_group)' not in anova_result.index else anova_result.loc['C(ad_type):C(age_group)', 'PR(>F)']
# 일부 버전에 따라 인덱스 명이 다를 수 있어 보정
if 'C(ad_type):C(age_group)' in anova_result.index:
    p_inter = anova_result.loc['C(ad_type):C(age_group)', 'PR(>F)']

print(f"1. 광고유형 유의성: {'[유의함]' if p_ad < 0.05 else '[유의하지 않음]'} (p={p_ad:.4e})")
print(f"2. 연령대 유의성  : {'[유의함]' if p_age < 0.05 else '[유의하지 않음]'} (p={p_age:.4e})")
print(f"3. 상호작용 유의성: {'[유의함]' if p_inter < 0.05 else '[유의하지 않음]'} (p={p_inter:.4f})")
print("=" * 50)

================ [이원분산분석 결과 보고서] =================
                           df       sum_sq      mean_sq           F  \
C(ad_type)                1.0   246.533333   246.533333   21.191977   
C(age_group)              2.0     1.866667     0.933333    0.080229   
C(ad_type):C(age_group)   2.0  2704.266667  1352.133333  116.229226   
Residual                 24.0   279.200000    11.633333         NaN   

                               PR(>F)  
C(ad_type)               1.138265e-04  
C(age_group)             9.231512e-01  
C(ad_type):C(age_group)  4.511601e-13  
Residual                          NaN  
--------------------------------------------------
1. 광고유형 유의성: [유의함] (p=1.1383e-04)
2. 연령대 유의성  : [유의하지 않음] (p=9.2315e-01)
3. 상호작용 유의성: [유의함] (p=0.0000)
